# 多代理协作：任务分解、共享状态与 A2A

用任务依赖和结果合同决定是否委派，验证冲突、预算与可回查证据，不以代理数量衡量能力。

Python 3.10+ / CPU / 无密钥。固定教学 fixture，不调用模型或服务。先读对应正文，预测成功与失败，再运行并修改一个边界。

In [ ]:
import math
def combine(results,budget):
    if type(budget) not in (int,float) or not math.isfinite(budget) or budget<0: raise ValueError('invalid budget')
    if any(type(r['cost']) not in (int,float) or not math.isfinite(r['cost']) or r['cost']<0 for r in results): raise ValueError('invalid cost')
    if sum(r['cost'] for r in results)>budget: raise ValueError('budget')
    claims={}
    for r in results:
        if r['status']!='completed': raise ValueError('incomplete subtask')
        for c in r['claims']:
            if not c['source']: raise ValueError('missing evidence')
            old=claims.get(c['key'])
            if old and old['value']!=c['value']: raise ValueError('conflict')
            sources=list(dict.fromkeys((old['sources'] if old else [])+[c['source']]))
            claims[c['key']]={**c,'sources':sources}
    return claims
a={'cost':2,'status':'completed','claims':[{'key':'version','value':'v1','source':'docs#version'}]}
b={'cost':3,'status':'completed','claims':[{'key':'persistence','value':'checkpoint','source':'docs#store'}]}
assert len(combine([a,b],5))==2
same=dict(a,claims=[dict(a['claims'][0],source='docs#another')])
assert combine([a,same],5)['version']['sources']==['docs#version','docs#another']
wrong={'cost':1,'status':'completed','claims':[{'key':'version','value':'v2','source':'old-blog'}]}
for inputs,limit in [([a,wrong],5),([a,b],4),([dict(a,cost=float('nan'))],5),([a],float('inf')),([dict(a,status='failed')],5),([dict(a,cost=-1)],5),([dict(a,claims=[dict(a['claims'][0],source='')])],5)]:
    try: combine(inputs,limit)
    except ValueError: pass
    else: raise AssertionError('bad merge accepted')
checks={'evidence_required':True,'conflict_rejected':True,'global_budget':True,'failure_retained':True}
print(checks)

## 记录与继续

在下一格 my_explanation 写下预测、改动与实际观察；运行后下载 mechanism_report.json，导回本篇笔记。下一次先独立解释边界，再对照旧记录。

In [ ]:
import json
from pathlib import Path
my_explanation = ''
report={'experiment':'lesson-multi-agent', 'scope':'stdlib fixtures; no model/service', 'checks':checks, 'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
